# GroupBy & Aggregation — answering "…by region", "…per product", "…for each month"

01 Core Python · **▶ 02 Pandas** · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies

`02_Pandas_Essentials/03_groupby_and_aggregation.ipynb`

---

### In one paragraph (no jargon)

Whenever a question contains the words **by**, **per** or **for each**, the answer is `groupby`. It works in three steps that pandas calls split–apply–combine: **split** the rows into piles by some column, **apply** a calculation to each pile, and **combine** the answers into a small table. "Total sales by branch", "average rating per product line", "how many customers in each region" — all the same three steps, all one line of code.

### After this notebook you can

- Group by one column and by several, and aggregate with sum, mean, count and more
- Apply different calculations to different columns in a single call with `.agg()`
- Understand the difference between `.agg()`, `.transform()` and `.filter()`
- Flatten the result back into an ordinary table an examiner can read

**Assumed knowledge:** `02_essential_operations.ipynb`

### What's inside

1. The split–apply–combine idea
2. Grouping by one column
3. The aggregation menu
4. Grouping by several columns
5. Different calculations per column with .agg()
6. ⚡ transform() and filter() — the two most under-used tools in pandas
7. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    return rebuild()


def rebuild_customer_data():
    """Recreate customer_data.csv (200 rows) with the same columns and behaviour."""
    rng = np.random.default_rng(42)          # fixed seed -> identical numbers every run
    n = 200
    frame = pd.DataFrame({
        'Customer_ID':     np.arange(1, n + 1),
        'Age':             rng.integers(18, 70, n).astype(float),
        'Gender':          rng.choice(['Male', 'Female'], n),
        'Income':          rng.normal(60000, 18000, n).round(-2).clip(20000, 150000),
        'Purchase_Amount': rng.gamma(4, 400, n).round(2),
        'Region':          rng.choice(['North', 'South', 'East', 'West'], n),
    })
    # the real file has a scattering of blanks — reproduce them so the cleaning code has work to do
    for col, frac in [('Age', .05), ('Income', .06), ('Purchase_Amount', .04)]:
        frame.loc[rng.choice(n, int(n * frac), replace=False), col] = np.nan
    return frame

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

Setup complete. pandas 3.0.2 | numpy 2.4.4


In [2]:
# The classic teaching frame — small enough to check the answers by hand
data = {'Company': ['GOOG', 'GOOG', 'MSFT', 'MSFT', 'FB', 'FB'],
        'Person':  ['Sam', 'Charlie', 'Amy', 'Vanessa', 'Carl', 'Sarah'],
        'Sales':   [200, 120, 340, 124, 243, 350]}
df = pd.DataFrame(data)

customers = load_data('customer_data.csv', rebuild=rebuild_customer_data)
display(df)

Loaded 'customer_data.csv' from /home/claude/work/build/Python-BDA-Complete-Notes/datasets


,Company,Person,Sales
0,GOOG,Sam,200
1,GOOG,Charlie,120
2,MSFT,Amy,340
3,MSFT,Vanessa,124
4,FB,Carl,243
5,FB,Sarah,350


## 1. The split–apply–combine idea

`df.groupby('Company')` on its own produces a *GroupBy object* — pandas has worked out the piles but hasn't
calculated anything yet, because you haven't said what to calculate. That's why printing it shows an obscure
`<...DataFrameGroupBy object at 0x...>` rather than a table. Add an aggregation and the answer appears.

In [3]:
grouped = df.groupby('Company')
print("On its own it's just a plan, not a result:")
print(" ", grouped, "\n")

print("Which rows landed in which pile:")
for company, rows in grouped:
    print(f"  {company}: rows {list(rows.index)}  sales {rows['Sales'].tolist()}")

print("\nNow ask for something — and the table appears:")
display(grouped[['Sales']].mean())

On its own it's just a plan, not a result:

Which rows landed in which pile:
  FB: rows [4, 5]  sales [243, 350]
  GOOG: rows [0, 1]  sales [200, 120]
  MSFT: rows [2, 3]  sales [340, 124]

Now ask for something — and the table appears:


,Sales
Company,
FB,296.5
GOOG,160.0
MSFT,232.0


## 2. Grouping by one column

`df.groupby('Company')['Sales'].mean()` reads left to right as: split by Company, take the Sales column, give
me the mean of each pile.

**A change worth knowing:** in current pandas, `df.groupby('Company').mean()` on a frame containing text
columns raises an error rather than silently skipping them. Either select the numeric column first (cleanest)
or pass `numeric_only=True`.

In [4]:
# The safe, explicit form: pick the column you want BEFORE aggregating
print("Mean sales per company:")
print(df.groupby('Company')['Sales'].mean(), "\n")

# Whole-frame aggregation with numeric_only — text columns are skipped rather than erroring
print("Whole frame, numeric columns only:")
display(df.groupby('Company').mean(numeric_only=True))

# The standard menu, one at a time
by_company = df.groupby('Company')['Sales']
summary = pd.DataFrame({
    'count': by_company.count(),   # how many rows
    'sum':   by_company.sum(),     # total
    'mean':  by_company.mean(),    # average
    'min':   by_company.min(),
    'max':   by_company.max(),
    'std':   by_company.std(),     # spread; NaN when a group has only one row
})
print("\nThe whole menu at once:")
display(summary.round(2))

Mean sales per company:
Company
FB      296.5
GOOG    160.0
MSFT    232.0
Name: Sales, dtype: float64 

Whole frame, numeric columns only:


,Sales
Company,
FB,296.5
GOOG,160.0
MSFT,232.0



The whole menu at once:


,count,sum,mean,min,max,std
Company,,,,,,
FB,2,593,296.5,243,350,75.66
GOOG,2,320,160.0,120,200,56.57
MSFT,2,464,232.0,124,340,152.74


In [5]:
# .describe() on a group gives every statistic for every pile
print("Full description per company:")
display(df.groupby('Company')['Sales'].describe())

# .transpose() (or .T) flips rows and columns — often far easier to read
print("\nTransposed — statistics down the side, companies across the top:")
display(df.groupby('Company')['Sales'].describe().transpose())

# …and now a single company is one clean column
print("\nJust GOOG:")
print(df.groupby('Company')['Sales'].describe().transpose()['GOOG'])

Full description per company:


,count,mean,std,min,25%,50%,75%,max
Company,,,,,,,,
FB,2.0,296.5,75.660426,243.0,269.75,296.5,323.25,350.0
GOOG,2.0,160.0,56.568542,120.0,140.00,160.0,180.00,200.0
MSFT,2.0,232.0,152.735065,124.0,178.00,232.0,286.00,340.0



Transposed — statistics down the side, companies across the top:


Company,FB,GOOG,MSFT
count,2.000000,2.000000,2.000000
mean,296.500000,160.000000,232.000000
std,75.660426,56.568542,152.735065
min,243.000000,120.000000,124.000000
25%,269.750000,140.000000,178.000000
50%,296.500000,160.000000,232.000000
75%,323.250000,180.000000,286.000000
max,350.000000,200.000000,340.000000



Just GOOG:
count      2.000000
mean     160.000000
std       56.568542
min      120.000000
25%      140.000000
50%      160.000000
75%      180.000000
max      200.000000
Name: GOOG, dtype: float64


## 3. Grouping by several columns

Pass a **list** of column names and pandas makes one pile per *combination*. The result has a MultiIndex — a
two-level row label. It's powerful but awkward to read, so `.reset_index()` at the end turns it back into an
ordinary flat table, which is what you want on an answer sheet.

In [6]:
print("Average spend by Region AND Gender:")
multi = customers.groupby(['Region', 'Gender'])['Purchase_Amount'].mean().round(2)
display(multi)

print("\nThe row labels are now a two-level MultiIndex:")
print(" ", multi.index[:3].tolist())

print("\n.reset_index() flattens it into a normal table — do this before presenting:")
display(multi.reset_index())

print("\n.unstack() moves the LAST grouping level into columns — a pivot table by another name:")
display(customers.groupby(['Region', 'Gender'])['Purchase_Amount'].mean().unstack().round(0))

Average spend by Region AND Gender:


Region  Gender
East    Female    2780.95
        Male      2465.52
North   Female    2431.25
        Male      2481.25
South   Female    3217.65
        Male      2511.11
West    Female    2490.91
        Male      2321.74
Name: Purchase_Amount, dtype: float64


The row labels are now a two-level MultiIndex:
  [('East', 'Female'), ('East', 'Male'), ('North', 'Female')]

.reset_index() flattens it into a normal table — do this before presenting:


,Region,Gender,Purchase_Amount
0,East,Female,2780.95
1,East,Male,2465.52
2,North,Female,2431.25
3,North,Male,2481.25
4,South,Female,3217.65
5,South,Male,2511.11
6,West,Female,2490.91
7,West,Male,2321.74



.unstack() moves the LAST grouping level into columns — a pivot table by another name:


Gender,Female,Male
Region,,
East,2781.0,2466.0
North,2431.0,2481.0
South,3218.0,2511.0
West,2491.0,2322.0


## 4. Different calculations per column with `.agg()`

`.agg()` is the one to remember. Hand it a dictionary of `{column: calculation}` and each column gets its own
treatment in a single pass. The **named-aggregation** form goes further and lets you choose the output column
names, which removes the messy two-level headers entirely.

In [7]:
# Dictionary form: one calculation (or several) per column
print("A different calculation for each column:")
display(customers.groupby('Region').agg({
    'Purchase_Amount': ['count', 'sum', 'mean'],
    'Income':          'median',
    'Age':             ['min', 'max'],
}).round(1))

print("\nNotice the two-level column headers — awkward. The named form fixes it:")

# NAMED AGGREGATION — output_name = (source_column, function). Clean, flat, self-documenting.
report = customers.groupby('Region').agg(
    customers      = ('Customer_ID',     'count'),
    total_spend    = ('Purchase_Amount', 'sum'),
    avg_spend      = ('Purchase_Amount', 'mean'),
    median_income  = ('Income',          'median'),
    youngest       = ('Age',             'min'),
    oldest         = ('Age',             'max'),
).round(0).sort_values('total_spend', ascending=False)

display(report)
print("\nThis is the shape an examiner wants: flat headers, meaningful names, sorted sensibly.")

A different calculation for each column:


Purchase_Amount                     Income   Age      
                 count       sum    mean   median   min   max
Region                                                       
East                55  142500.0  2590.9  50000.0  19.0  58.0
North               37   88500.0  2391.9  55000.0  18.0  56.0
South               40  112500.0  2812.5  50000.0  18.0  58.0
West                54  134800.0  2496.3  55000.0  18.0  59.0


Notice the two-level column headers — awkward. The named form fixes it:


,customers,total_spend,avg_spend,median_income,youngest,oldest
Region,,,,,,
East,56,142500.0,2591.0,50000.0,19.0,58.0
West,59,134800.0,2496.0,55000.0,18.0,59.0
South,40,112500.0,2812.0,50000.0,18.0,58.0
North,38,88500.0,2392.0,55000.0,18.0,56.0



This is the shape an examiner wants: flat headers, meaningful names, sorted sensibly.


In [8]:
# Your own function inside .agg() — anything that takes a Series and returns one number
def spend_range(s):
    """Gap between the biggest and smallest purchase in the group."""
    return s.max() - s.min()

display(customers.groupby('Region').agg(
    avg      = ('Purchase_Amount', 'mean'),
    spread   = ('Purchase_Amount', spend_range),          # your function, by name
    top_10pc = ('Purchase_Amount', lambda s: s.quantile(0.9)),   # or a lambda
    n_male   = ('Gender', lambda s: (s == 'Male').sum()),        # a conditional count
).round(1))

# Counting rows per group — three ways, subtly different
print("\n.size()  counts rows including blanks :", customers.groupby('Region').size().to_dict())
print(".count() counts NON-BLANK values       :", customers.groupby('Region')['Income'].count().to_dict())
print("value_counts() — the shortcut for pure counts:")
print(customers['Region'].value_counts().to_string())

,avg,spread,top_10pc,n_male
Region,,,,
East,2590.9,4300.0,4160.0,30
North,2391.9,4400.0,4560.0,17
South,2812.5,4300.0,4010.0,18
West,2496.3,4200.0,4170.0,27



.size()  counts rows including blanks : {'East': 56, 'North': 38, 'South': 40, 'West': 59}
.count() counts NON-BLANK values       : {'East': 51, 'North': 32, 'South': 39, 'West': 54}
value_counts() — the shortcut for pure counts:
Region
West     59
East     56
South    40
North    38


### ⚡ Beyond the syllabus — `transform()` and `filter()` — the two most under-used tools in pandas

`.agg()` collapses each group to one row. But often you want the group's answer **written back onto every row** — "how does this customer compare to their region's average?". That's `.transform()`, and it's how you compute percentage-of-group, group-relative z-scores, and running totals without a single loop or merge. `.filter()` is its partner: it keeps or discards *whole groups* based on a test.

In [9]:
# ---- transform(): group result broadcast back to every row --------------------
work = customers.dropna(subset=['Purchase_Amount']).copy()

# agg gives 4 rows (one per region); transform gives 200 (one per customer)
print("groupby().mean()      ->", customers.groupby('Region')['Purchase_Amount'].mean().shape, "rows")
print("groupby().transform() ->", work.groupby('Region')['Purchase_Amount'].transform('mean').shape, "rows\n")

work['region_avg']   = work.groupby('Region')['Purchase_Amount'].transform('mean')
work['vs_region_pc'] = (work['Purchase_Amount'] / work['region_avg'] - 1) * 100
work['region_total'] = work.groupby('Region')['Purchase_Amount'].transform('sum')
work['share_of_region_pc'] = work['Purchase_Amount'] / work['region_total'] * 100

# Group-relative z-score: how unusual is this customer WITHIN their own region?
work['region_z'] = work.groupby('Region')['Purchase_Amount'].transform(
    lambda s: (s - s.mean()) / s.std()
)

display(work[['Customer_ID', 'Region', 'Purchase_Amount', 'region_avg',
              'vs_region_pc', 'share_of_region_pc', 'region_z']].head(6).round(2))

print("\nNow a question like 'which customers spend more than twice their region average?'")
print("is a plain filter — no merge, no loop:")
display(work[work['Purchase_Amount'] > 2 * work['region_avg']]
        [['Customer_ID', 'Region', 'Purchase_Amount', 'region_avg']].head())

groupby().mean()      -> (4,) rows
groupby().transform() -> (193,) rows



,Customer_ID,Region,Purchase_Amount,region_avg,vs_region_pc,share_of_region_pc,region_z
0,1,West,2400.0,2496.30,-3.86,1.78,-0.08
1,2,North,1900.0,2391.89,-20.56,2.15,-0.39
2,3,East,2400.0,2590.91,-7.37,1.68,-0.15
3,4,NaN,1400.0,NaN,NaN,NaN,NaN
4,5,East,4000.0,2590.91,54.39,2.81,1.11
5,6,West,3900.0,2496.30,56.23,2.89,1.19



Now a question like 'which customers spend more than twice their region average?'
is a plain filter — no merge, no loop:


,Customer_ID,Region,Purchase_Amount,region_avg
56,57,North,4800.0,2391.891892
93,94,North,4900.0,2391.891892
187,188,North,4900.0,2391.891892
191,192,North,4800.0,2391.891892


In [10]:
# ---- filter(): keep or drop WHOLE groups -------------------------------------
# "Only analyse regions with more than 45 customers"
big_regions = customers.groupby('Region').filter(lambda g: len(g) > 45)
print("Region sizes:", customers['Region'].value_counts().to_dict())
print("Kept after filter(len > 45):", big_regions['Region'].unique().tolist())

# "Only regions whose average spend beats the national average"
national_avg = customers['Purchase_Amount'].mean()
strong = customers.groupby('Region').filter(lambda g: g['Purchase_Amount'].mean() > national_avg)
print(f"\nNational average spend: {national_avg:.0f}")
print("Regions beating it:", strong['Region'].unique().tolist())

# ---- The complete pattern: group, rank within group, take the top from each -----
print("\nTop 2 spenders in every region (a classic exam question):")
top_per_region = (work
    .assign(rank_in_region=work.groupby('Region')['Purchase_Amount']
                               .rank(ascending=False, method='first'))
    .query('rank_in_region <= 2')
    .sort_values(['Region', 'rank_in_region'])
    [['Region', 'rank_in_region', 'Customer_ID', 'Purchase_Amount']])
display(top_per_region)

Region sizes: {'West': 59, 'East': 56, 'South': 40, 'North': 38}
Kept after filter(len > 45): ['West', 'East']

National average spend: 2559
Regions beating it: ['East', 'South']

Top 2 spenders in every region (a classic exam question):


,Region,rank_in_region,Customer_ID,Purchase_Amount
95,East,1.0,96,4800.0
154,East,2.0,155,4600.0
93,North,1.0,94,4900.0
187,North,2.0,188,4900.0
47,South,1.0,48,4900.0
197,South,2.0,198,4600.0
111,West,1.0,112,4700.0
138,West,2.0,139,4500.0


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Group and average | `df.groupby('c')['v'].mean()` |
| Group and total | `df.groupby('c')['v'].sum()` |
| Rows per group | `df.groupby('c').size()` |
| Whole frame | `df.groupby('c').mean(numeric_only=True)` |
| Every statistic | `df.groupby('c')['v'].describe()` |
| Two grouping columns | `df.groupby(['a','b'])['v'].sum()` |
| Flatten the result | `… .reset_index()` |
| Groups across the top | `… .unstack()` |
| Different calc per column | `df.groupby('c').agg({'v':'sum','w':'mean'})` |
| Named output columns | `.agg(total=('v','sum'))` |
| Group value on every row | `df.groupby('c')['v'].transform('mean')` |
| Keep whole groups | `df.groupby('c').filter(lambda g: len(g) > 5)` |
| Rank inside a group | `df.groupby('c')['v'].rank(ascending=False)` |
| Sort the summary | `… .sort_values('total', ascending=False)` |

### Adapting this in the exam

- The words 'by', 'per' or 'for each' in a question → `groupby`.
- '…as a percentage of the group total' → `transform('sum')` then divide.
- 'Top N within each group' → `groupby(...).rank()` then filter, or `groupby(...).head(N)` after sorting.
- Result looks like a jumble of indented labels? You have a MultiIndex — add `.reset_index()` or `.unstack()`.

### Traps that cost marks

- `df.groupby('c')` alone prints an object, not a table. It needs an aggregation to produce anything.
- `.mean()` on a frame with text columns errors in current pandas. Select the numeric column first, or pass `numeric_only=True`.
- Grouping produces a MultiIndex when you group by several columns — `.reset_index()` before presenting or merging.
- `.size()` counts rows (blanks included); `.count()` counts non-blank values. They differ exactly where your data is missing.
- `.agg()` returns one row per group; `.transform()` returns one row per **original** row. Using the wrong one gives a shape mismatch.
- By default `groupby` **drops rows where the grouping column is blank**. Pass `dropna=False` if those rows matter.